In [8]:
import json, os
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
from open_clip import create_model_from_pretrained, get_tokenizer

DEVICE     = "cuda"
IMAGE_DIR  = "/shared/rc/vlm-idk/LLaVA/data/images/"
ANS_PATH   = "/home/as5606/projects/Super_LlaVA_Med/medium_answerable.jsonl"
UNANS_PATH = "/home/as5606/projects/Super_LlaVA_Med/medium_unanswerable.jsonl"

model, preprocess = create_model_from_pretrained(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224', device=DEVICE
)
tokenizer = get_tokenizer(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)
model.eval()


CustomTextCLIP(
  (visual): TimmModel(
    (trunk): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Attention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): Identity()
          (drop_path1): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          

In [9]:
def get_correlation_vector(img_id, statement):
    img_path = os.path.join(IMAGE_DIR, f"{img_id}.jpg")
    if not os.path.exists(img_path):
        return None

    img = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(DEVICE)
    tok = tokenizer([statement]).to(DEVICE)

    with torch.no_grad():
        img_feat  = model.encode_image(img)
        img_feat  = img_feat / img_feat.norm(dim=-1, keepdim=True)
        text_feat = model.encode_text(tok)
        text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

    return (img_feat * text_feat).squeeze(0).cpu()  # [512]


def load_and_encode(jsonl_path, label):
    vectors, labels = [], []
    entries = [json.loads(l) for l in open(jsonl_path)]
    for rec in tqdm(entries, desc=f"Encoding {label}"):
        vec = get_correlation_vector(rec["id"], rec["statement"])
        if vec is not None:
            vectors.append(vec)
            labels.append(label)
    return vectors, labels

pos_vecs, pos_labels = load_and_encode(ANS_PATH,   label=1)  # answerable   = 1
neg_vecs, neg_labels = load_and_encode(UNANS_PATH, label=0)  # unanswerable = 0

all_vecs   = torch.stack(pos_vecs + neg_vecs)         # [N, 512]
all_labels = torch.tensor(pos_labels + neg_labels, dtype=torch.float32)  # [N]

print(f"Total: {len(all_labels)}  |  Answerable: {int(all_labels.sum())}  |  Unanswerable: {int((1-all_labels).sum())}")


Encoding 0: 100%|█████████████████████████████████████████████████████████████████████████| 2141/2141 [00:43<00:00, 49.40it/s]

Total: 4282  |  Answerable: 2141  |  Unanswerable: 2141


In [10]:
from sklearn.model_selection import train_test_split

idx = list(range(len(all_labels)))
train_idx, test_idx = train_test_split(idx, test_size=0.2, stratify=all_labels.numpy(), random_state=42)

X_train = all_vecs[train_idx].to(DEVICE)
y_train = all_labels[train_idx].to(DEVICE)
X_test  = all_vecs[test_idx].to(DEVICE)
y_test  = all_labels[test_idx].to(DEVICE)

print(f"Train: {len(train_idx)}  |  Test: {len(test_idx)}")


Train: 3425  |  Test: 857


In [11]:
import torch.nn as nn

# single linear layer: 512 → 1 (exactly like CLIP-UP's LinearSignalProjection)
projection = nn.Linear(512, 1).to(DEVICE)

optimizer = torch.optim.Adam(projection.parameters(), lr=1e-3)
loss_fn   = nn.BCEWithLogitsLoss()

EPOCHS     = 50
BATCH_SIZE = 64

dataset    = torch.utils.data.TensorDataset(X_train, y_train)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

projection.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for xb, yb in dataloader:
        optimizer.zero_grad()
        logits = projection(xb).squeeze(1)
        loss   = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS}  loss: {total_loss/len(dataloader):.4f}")


Epoch  10/50  loss: 0.6897
Epoch  20/50  loss: 0.6862
Epoch  30/50  loss: 0.6828
Epoch  40/50  loss: 0.6794
Epoch  50/50  loss: 0.6760


In [12]:
from sklearn.metrics import classification_report, confusion_matrix

projection.eval()
with torch.no_grad():
    logits = projection(X_test).squeeze(1)
    probs  = torch.sigmoid(logits).cpu().numpy()

y_pred = (probs >= 0.5).astype(int)
y_true = y_test.cpu().numpy().astype(int)

print(classification_report(y_true, y_pred, target_names=["Unanswerable", "Answerable"]))
print("Confusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_true, y_pred))


              precision    recall  f1-score   support

Unanswerable       0.78      0.65      0.71       429
  Answerable       0.70      0.82      0.76       428

    accuracy                           0.73       857
   macro avg       0.74      0.73      0.73       857
weighted avg       0.74      0.73      0.73       857

Confusion matrix (rows=actual, cols=predicted):
[[277 152]
 [ 76 352]]


In [7]:
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# load text only — no images
ans_entries   = [json.loads(l) for l in open("/home/as5606/projects/Super_LlaVA_Med/medium_answerable.jsonl")]
unans_entries = [json.loads(l) for l in open("/home/as5606/projects/Super_LlaVA_Med/medium_unanswerable.jsonl")]

texts  = [e["statement"] for e in ans_entries] + [e["statement"] for e in unans_entries]
labels = [1] * len(ans_entries) + [0] * len(unans_entries)  # 1=answerable, 0=unanswerable

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, stratify=labels, random_state=42
)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000, sublinear_tf=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)

y_pred = clf.predict(X_test_vec)
print(classification_report(y_test, y_pred, target_names=["Unanswerable", "Answerable"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

Unanswerable       0.44      0.48      0.46       429
  Answerable       0.43      0.40      0.42       428

    accuracy                           0.44       857
   macro avg       0.44      0.44      0.44       857
weighted avg       0.44      0.44      0.44       857

Confusion matrix:
[[204 225]
 [255 173]]
